In [3]:
import json
import spacy
from spacy.tokens import DocBin
from tqdm import tqdm
import re

nlp = spacy.blank("en")
doc_bin = DocBin()

file_path = "data/resumes/resumes_dataset.jsonl"
save_path = "data/resumes/train.spacy"

print("Auto-labeling skills and generating binary data...")

with open(file_path, 'r', encoding='utf-8') as f:
    for line in tqdm(f.readlines()[:500]): 
        data = json.loads(line)
        
        # Extract the raw text and the comma-separated skills
        text = data.get('Text', '')
        skills_raw = data.get('Skills', '')
        
        if not text or not skills_raw:
            continue
            
        doc = nlp.make_doc(text)
        ents = []
        
        # Clean up the list of skills
        skill_list = [s.strip() for s in skills_raw.split(',') if s.strip()]
        
        # Automatically search the text to find the start and end positions of each skill
        for skill in set(skill_list):
            # Using regex to find all mentions of the skill in the text
            for match in re.finditer(re.escape(skill), text, flags=re.IGNORECASE):
                span = doc.char_span(match.start(), match.end(), label="SKILL", alignment_mode="contract")
                if span is not None:
                    ents.append(span)
        
        # Filter out overlapping labels (e.g., if it finds "Java" inside "JavaScript")
        doc.ents = spacy.util.filter_spans(ents)
        
        # CRITICAL FIX: Only save the document if we successfully found and labeled skills in it
        if len(doc.ents) > 0:
            doc_bin.add(doc)

doc_bin.to_disk(save_path)
print(f"Success! Generated valid training data.")

Auto-labeling skills and generating binary data...


100%|████████████████████████████████████████| 500/500 [00:00<00:00, 801.89it/s]


Success! Generated valid training data.
